# Giai đoạn 2 — Mục 2.1, 2.3, 2.4 — Huấn luyện và đánh giá CNN raw và envelope với LOLO
**Đầu ra**:
- `outputs/tables/cnn_raw_lolo_results.csv`
- `outputs/tables/cnn_env_lolo_results.csv`
- Các model `.h5` và `.tflite` trong `outputs/models/`

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from tensorflow.keras.callbacks import EarlyStopping

from common import models, quantization, training

In [4]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

windows_raw_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_raw.parquet")
windows_env_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_env.parquet")

In [16]:
manifest_filtered = pd.read_csv("../giai_doan_1_tien_xu_ly/outputs/tables/manifest_filtered.csv")

# Hàm trích xuất dữ liệu cho CNN

In [5]:
def get_cnn_data(df, window_col='window'):
    X = np.stack(df[window_col].values).astype(np.float32)
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True) + 1e-8
    X = (X - mean) / std
    X = X[..., np.newaxis]
    y = df['label'].values
    return X, y

# 1DCNN (raw/env) - LOLO

In [6]:
def run_cnn_lolo(df, window_col, build_model_func, window_size, model_name):
    results = []
    for fold_info, train_df, val_df, test_df in training.iterate_lolo_splits(df, load_col='load_hp'):
        print(f"\n--- {model_name} - Fold: {fold_info['fold_name']} ---")
        
        X_train, y_train = get_cnn_data(train_df, window_col)
        X_val, y_val = get_cnn_data(val_df, window_col)
        X_test, y_test = get_cnn_data(test_df, window_col)
        
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_val_enc = le.transform(y_val)
        y_test_enc = le.transform(y_test)
        
        model = build_model_func(window_size=window_size)
        model = models.compile_classifier(model)
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        history = model.fit(X_train, y_train_enc,
                            validation_data=(X_val, y_val_enc),
                            epochs=100, batch_size=64,
                            callbacks=[early_stop], verbose=0)
        
        loss, acc = model.evaluate(X_test, y_test_enc, verbose=0)
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        f1 = f1_score(y_test_enc, y_pred, average='macro')
        
        tflite_bytes = quantization.quantize_model_int8(model, X_train)
        int8_result = quantization.evaluate_tflite_model(tflite_bytes, X_test, y_test_enc)
        
        results.append({
            'fold': fold_info['fold_name'],
            'test_load': fold_info['test_load'],
            'float_accuracy': acc,
            'float_f1': f1,
            'int8_accuracy': int8_result['accuracy'],
            'epochs': len(history.history['loss'])
        })
        
        model.save(MODELS_DIR / f"{model_name}_{fold_info['fold_name']}.h5")
        quantization.model_bytes_to_file(tflite_bytes, MODELS_DIR / f"{model_name}_{fold_info['fold_name']}.tflite")
    
    return pd.DataFrame(results)

In [19]:
print("Các cột của manifest_filtered:", manifest_filtered.columns)
print("Index của manifest_filtered:", manifest_filtered.index.name)
display(manifest_filtered.head(2))

Các cột của manifest_filtered: Index(['file_path', 'load_hp', 'label', 'fault_diameter_mils', 'or_position',
       'source_category', 'sensor_location', 'declared_sample_rate_khz',
       'n_samples_DE', 'n_samples_FE', 'n_samples_BA', 'rpm_from_file',
       'read_error', 'warnings', 'has_warning'],
      dtype='str')
Index của manifest_filtered: None


,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,NaN,NaN,False
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,NaN,NaN,False


In [18]:
# Lấy file_id và load_hp từ bảng manifest gốc (hoặc bảng nào chứa thông tin file của bạn)
metadata = manifest_filtered[['file_id', 'load_hp']]

# Ghép (merge) cột load_hp vào 2 bảng window dựa trên 'file_id'
windows_raw_df = windows_raw_df.merge(metadata, on='file_id', how='left')
windows_env_df = windows_env_df.merge(metadata, on='file_id', how='left')

# In ra kiểm tra lại xem đã có cột load_hp chưa
print("Các cột của windows_raw_df sau khi merge:", windows_raw_df.columns)

KeyError: "['file_id'] not in index"

In [9]:
cnn_raw_results = run_cnn_lolo(windows_raw_df, 'window', models.build_cnn1d_raw, 2048, 'cnn_raw')
cnn_env_results = run_cnn_lolo(windows_env_df, 'window', models.build_cnn1d_env, 1024, 'cnn_env')

cnn_raw_results.to_csv(TABLES_DIR / "cnn_raw_lolo_results.csv", index=False)
cnn_env_results.to_csv(TABLES_DIR / "cnn_env_lolo_results.csv", index=False)

KeyError: 'load_hp'